# PCAOB Inspection Results and Auditor Workload
**AC 547 project: data build and first-pass analysis**

**Research question (draft):** How have PCAOB Part I.A deficiency rates changed over time across audit firm tiers, and are firms whose engagement partners carry more public company clients associated with higher deficiency rates?

**Data sources (both public, from pcaobus.org):**
1. `inspection-reports-json.json`: Firm Inspection Reports dataset ("Firm Data and Audits Selected for Review"), one row per inspection report.
2. `FirmFilings.zip`: AuditorSearch (Form AP) full download, one row per Form AP filing.

**Design choices made so far (state these in the methods section):**
- Keep only regular issuer audits (drop investment companies and employee benefit plans).
- Keep only the latest version of each Form AP (drop superseded amendments).
- Match inspection year *Y* to Form AP fiscal year *Y − 1*, because an inspection reviews the prior year's audits.
- Join the two sources on PCAOB firm ID, not firm name.
- Measure workload the same way for every firm: unique issuer clients ÷ unique signing engagement partners, from Form AP.

**Privacy note:** Form AP contains individual partner names and contact details. This notebook never loads those columns. Partners are counted by their ID number only.

## 1. Setup
Change `DATA_DIR` to the folder where you saved the two downloads.

In [ ]:
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "."   # <-- change this to your folder
INSPECTION_FILE = f"{DATA_DIR}/inspection-reports-json.json"
FORM_AP_ZIP = f"{DATA_DIR}/FirmFilings.zip"

FIRST_INSPECTION_YEAR = 2018   # first year the detailed PCAOB data sets cover
LAST_INSPECTION_YEAR = 2025    # see note in Section 2 about 2025 being incomplete

# PCAOB firm IDs for the U.S. Big 4 (verified against both files)
BIG4_IDS = {42: "Ernst & Young LLP", 34: "Deloitte & Touche LLP",
            238: "PricewaterhouseCoopers LLP", 185: "KPMG LLP"}

pd.set_option("display.width", 140)

## 2. Load and clean the inspection data
Every field in the JSON file arrives as text, so numeric fields are converted first.

We also **recompute** the deficiency rate as deficient audits ÷ audits reviewed. That gives full precision (PCAOB's field is rounded to a whole percent) and lets us check that the published figure agrees with its parts.

In [ ]:
with open(INSPECTION_FILE) as f:
    ins = pd.DataFrame(json.load(f))

num_cols = ["InspectionYear", "AuditsWithPartIADeficiencies", "TotalAuditsReviewed",
            "PartIADeficiencyRate", "TotalIssuerAuditClients",
            "TotalEngagementPartnersOnIssuerAuditWork"]
for c in num_cols:
    ins[c] = pd.to_numeric(ins[c], errors="coerce")

ins = ins.rename(columns={"RegistrationId": "firm_id", "InspectionYear": "insp_year"})
print("All inspection reports:", len(ins))

ins = ins[ins["insp_year"].between(FIRST_INSPECTION_YEAR, LAST_INSPECTION_YEAR)].copy()
print(f"Reports in {FIRST_INSPECTION_YEAR}-{LAST_INSPECTION_YEAR}:", len(ins))

# Recompute the deficiency rate and compare to PCAOB's rounded figure
ins["def_rate"] = ins["AuditsWithPartIADeficiencies"] / ins["TotalAuditsReviewed"]
gap = (ins["def_rate"] * 100).round() - ins["PartIADeficiencyRate"]
print("Reports where recomputed rate differs from published rate by >1 point:",
      int((gap.abs() > 1).sum()))

# One report per firm per year?
print("Duplicate firm-year reports:", int(ins.duplicated(["firm_id", "insp_year"]).sum()))

ins.groupby("insp_year").size().rename("reports")

**Data-quality note:** the most recent inspection year is incomplete because PCAOB issues reports on a rolling basis. Check the report counts above. If the last year has far fewer reports than earlier years, consider dropping it or labeling it as partial.

## 3. Assign firm tiers
Following the professor's suggestion:
- **Big 4**: the four U.S. Big 4 firms, by firm ID
- **Other annual**: other annually inspected firms (more than 100 issuer audit clients)
- **Triennial**: firms inspected at least every three years (100 or fewer)

Foreign Big 4 network affiliates are inspected triennially, so they fall in the Triennial tier. Decide as a group whether to keep non-U.S. firms at all (see Section 8).

In [ ]:
def tier(row):
    if row["firm_id"] in BIG4_IDS:
        return "Big 4"
    if row["InspectionType"] == "Annually Inspected":
        return "Other annual"
    return "Triennial"

ins["tier"] = ins.apply(tier, axis=1)
pd.crosstab(ins["tier"], ins["Country"].eq("United States").map({True: "U.S.", False: "Non-U.S."}))

## 4. Load and clean Form AP
Only the columns we need are loaded, which skips all name and contact fields.

Steps:
1. Keep latest filings only (`Latest Form AP Filing == 1`).
2. Keep regular issuer audits only.
3. Parse the fiscal period end date. **Note:** the data dictionary says mm/dd/yyyy, but the file actually stores `m/d/yyyy h:mm:ss AM`. The format below matches the real file.

In [ ]:
AP_COLS = ["Form Filing ID", "Latest Form AP Filing", "Firm ID", "Audit Report Type",
           "Issuer ID", "Issuer CIK", "Fiscal Period End Date", "Engagement Partner ID"]

with zipfile.ZipFile(FORM_AP_ZIP) as z:
    csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    ap = pd.read_csv(z.open(csv_name), usecols=AP_COLS, dtype=str)

print("All Form AP filings:", len(ap))
print(ap["Audit Report Type"].value_counts(), "\n")

ap = ap[ap["Latest Form AP Filing"] == "1"]
print("After dropping superseded amendments:", len(ap))

ap = ap[ap["Audit Report Type"] == "Issuer, other than Employee Benefit Plan or Investment Company"]
print("After keeping regular issuers only:", len(ap))

ap["fiscal_year"] = pd.to_datetime(ap["Fiscal Period End Date"],
                                   format="%m/%d/%Y %I:%M:%S %p", errors="coerce").dt.year
print("Unparseable dates:", int(ap["fiscal_year"].isna().sum()))
print("Implausible fiscal years (before 2014):", int((ap["fiscal_year"] < 2014).sum()))

ap["firm_id"] = pd.to_numeric(ap["Firm ID"])

# Client identifier: some filings are missing the PCAOB Issuer ID.
# Fall back to the SEC CIK, and if both are missing, treat the filing as its own client.
print("Missing Issuer ID:", int(ap["Issuer ID"].isna().sum()),
      "| missing both Issuer ID and CIK:", int((ap["Issuer ID"].isna() & ap["Issuer CIK"].isna()).sum()))
ap["client_key"] = ("ID" + ap["Issuer ID"]).fillna("CIK" + ap["Issuer CIK"]).fillna("FILING" + ap["Form Filing ID"])
ap["fiscal_year"].value_counts().sort_index().loc[2014:]

## 5. Build the workload measure
For each firm and fiscal year:
- **clients** = number of unique issuers (by `client_key`: Issuer ID, else CIK)
- **partners** = number of unique signing engagement partners (by `Engagement Partner ID`)
- **clients_per_partner** = clients ÷ partners

Then each inspection year *Y* is matched to fiscal year *Y − 1*.

**Limitation to disclose:** a filing's fiscal year is the year its fiscal period *ended*. Companies with non-December year-ends make the one-year lag approximate.

In [ ]:
workload = (ap.groupby(["firm_id", "fiscal_year"])
              .agg(clients=("client_key", "nunique"),
                   partners=("Engagement Partner ID", "nunique"))
              .reset_index())
workload["clients_per_partner"] = workload["clients"] / workload["partners"]

ins["fiscal_year"] = ins["insp_year"] - 1
df = ins.merge(workload, on=["firm_id", "fiscal_year"], how="left")

print("Inspection reports with a Form AP workload value, by tier:")
pd.crosstab(df["tier"], df["clients"].notna().map({True: "matched", False: "no Form AP match"}))

Unmatched reports are most likely firms that did not sign any issuer audit reports in that fiscal year (for example, firms that only assisted on other firms' audits). Report the number dropped.

## 6. Validate against firms' self-reported counts
The inspection file includes self-reported client and partner counts, **but only for triennial firms**. Where both exist, compare them.

Expect client counts to agree closely. Partner counts should agree less, because the inspection file counts *all* partners on issuer work, while Form AP counts only partners who *sign*.

In [ ]:
v = df[df["clients"].notna()
       & df["TotalIssuerAuditClients"].notna()
       & df["TotalEngagementPartnersOnIssuerAuditWork"].notna()].copy()

print("Reports available for validation:", len(v))
print("Correlation, clients:  %.2f" % v[["TotalIssuerAuditClients", "clients"]].corr().iloc[0, 1])
print("Correlation, partners: %.2f" % v[["TotalEngagementPartnersOnIssuerAuditWork", "partners"]].corr().iloc[0, 1])
print("Median difference (self-reported minus Form AP): clients %.1f, partners %.1f" % (
    (v["TotalIssuerAuditClients"] - v["clients"]).median(),
    (v["TotalEngagementPartnersOnIssuerAuditWork"] - v["partners"]).median()))

## 7. Build the analysis sample
Drop reports that cannot be analyzed and record how many were dropped at each step. This becomes the sample-selection table in your paper.

In [ ]:
steps = [("Inspection reports in period", len(df))]

s = df[df["clients"].notna()]
steps.append(("Has a Form AP workload match", len(s)))

s = s[s["TotalAuditsReviewed"] > 0]
steps.append(("At least one audit reviewed", len(s)))

s = s.copy()
s["log_cpp"] = np.log(s["clients_per_partner"])   # log scale tames extreme values

sample_table = pd.DataFrame(steps, columns=["Step", "Reports remaining"])
sample_table

## 8. Descriptive statistics
Report medians alongside means, because a few firms have extreme workloads.

In [ ]:
s.groupby("tier").agg(
    reports=("firm_id", "size"),
    firms=("firm_id", "nunique"),
    mean_def_rate=("def_rate", "mean"),
    median_def_rate=("def_rate", "median"),
    median_cpp=("clients_per_partner", "median"),
    max_cpp=("clients_per_partner", "max"),
).round(3)

In [ ]:
# Largest workloads: check these by hand before drawing conclusions
s.sort_values("clients_per_partner", ascending=False)[
    ["Company", "Country", "tier", "insp_year", "clients", "partners", "clients_per_partner", "def_rate"]
].head(10)

**Group decision still open:** keep or drop non-U.S. firms? Foreign firms face different legal environments, and PCAOB access to some jurisdictions has been limited. Restricting to U.S. firms is cleaner. Keeping them gives a larger sample. Whichever you choose, state it as a design choice.

## 9. Trends over time by tier
**Interpretation caution:** a tier's rate in a given year is pooled across firms. Triennial firms appear only in their inspection years, so each year's triennial group is a *different* set of firms.

In [ ]:
trend = (s.groupby(["insp_year", "tier"])
           .agg(deficient=("AuditsWithPartIADeficiencies", "sum"),
                reviewed=("TotalAuditsReviewed", "sum"))
           .reset_index())
# Pooled rate = total deficient audits / total audits reviewed (weights larger inspections more)
trend["pooled_rate"] = trend["deficient"] / trend["reviewed"]

fig, ax = plt.subplots(figsize=(8, 4.5))
for t, g in trend.groupby("tier"):
    ax.plot(g["insp_year"], g["pooled_rate"] * 100, marker="o", label=t)
ax.set_xlabel("Inspection year")
ax.set_ylabel("Part I.A deficiency rate (%)")
ax.set_title("Pooled Part I.A deficiency rate by firm tier")
ax.legend()
plt.tight_layout()
plt.show()

trend.pivot(index="insp_year", columns="tier", values="pooled_rate").round(3)

## 10. Workload vs. deficiency rate
Scatter of log clients-per-partner against the deficiency rate, by tier. Small triennial inspections (often 1–3 audits reviewed) produce rates of exactly 0%, 33%, 50% or 100%, which is the volatility the professor warned about.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for t, g in s.groupby("tier"):
    ax.scatter(g["log_cpp"], g["def_rate"] * 100, alpha=0.5, label=t, s=18)
ax.set_xlabel("log(clients per signing partner)")
ax.set_ylabel("Part I.A deficiency rate (%)")
ax.set_title("Auditor workload vs. inspection deficiency rate")
ax.legend()
plt.tight_layout()
plt.show()

for t, g in s.groupby("tier"):
    print(f"{t:13s} n={len(g):4d}  Spearman correlation = {g[['log_cpp','def_rate']].corr(method='spearman').iloc[0,1]:.2f}")

## 11. Starter regression (optional)
Requires `statsmodels` (`pip install statsmodels`).

**Model:** deficiency rate on log workload, tier and inspection-year dummies. Weighted by audits reviewed, so a 1-audit inspection doesn't count as much as a 60-audit one. Standard errors are clustered by firm, because the same firm appears in multiple years.

**What it can and can't show:** an association only. PCAOB selects audits mostly on risk, so the deficiency rate reflects the riskiest engagements, not the firm's whole practice. Workload may also stand in for firm characteristics we can't observe.

A fractional-response (logit) model is a stronger alternative for a 0–1 outcome. Ask your professor whether weighted OLS is acceptable for this course.

In [ ]:
try:
    import statsmodels.formula.api as smf
    reg = s.dropna(subset=["def_rate", "log_cpp"]).copy()
    model = smf.wls("def_rate ~ log_cpp + C(tier) + C(insp_year)",
                    data=reg, weights=reg["TotalAuditsReviewed"])
    result = model.fit(cov_type="cluster", cov_kwds={"groups": reg["firm_id"]})
    print(result.summary())
except ImportError:
    print("statsmodels not installed. Run: pip install statsmodels")

## 12. Save the cleaned firm-year file
This file contains firm-level data only (no partner or issuer details).

In [ ]:
keep = ["firm_id", "Company", "Country", "tier", "insp_year", "fiscal_year",
        "TotalAuditsReviewed", "AuditsWithPartIADeficiencies", "def_rate",
        "clients", "partners", "clients_per_partner", "log_cpp"]
s[keep].to_csv(f"{DATA_DIR}/firm_year_analysis.csv", index=False)
print("Saved", len(s), "rows to firm_year_analysis.csv")